# Day 0: Generate the Embeddings

This notebook embeds the Nordvik crime dataset using `all-MiniLM-L6-v2` and saves the results to disk. Every chapter notebook loads these files directly - we only need to run this once.

**Inputs**
- `data/nordvik_crimes.parquet` - generated by `generate_dataset.ipynb`

**Outputs**
- `data/nordvik_embeddings.npy` - float32 array of shape `(n_records, 384)`
- `data/nordvik_ids.npy` - string array of IncidentIDs in matching row order

## 1. Install Dependencies

In [1]:
%pip install numpy==2.2.6 \
             pandas==3.0.3 \
             pyarrow==25.0.0 \
             sentence-transformers==5.6.1 \
             tqdm==4.68.3 --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [2]:
import numpy as np
import os
import pandas as pd
import time

from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

## 3. Configuration

In [3]:
DATA_DIR        = "data"
PARQUET_PATH    = os.path.join(DATA_DIR, "nordvik_crimes.parquet")
EMBEDDINGS_PATH = os.path.join(DATA_DIR, "nordvik_embeddings.npy")
IDS_PATH        = os.path.join(DATA_DIR, "nordvik_ids.npy")

MODEL_NAME  = "all-MiniLM-L6-v2"
BATCH_SIZE  = 256

In [4]:
assert os.path.exists(PARQUET_PATH), f"ERROR  {PARQUET_PATH} not found - run generate_dataset.ipynb first"

## 4. Load Dataset

In [5]:
df = pd.read_parquet(PARQUET_PATH)

print(f"OK  Loaded {len(df):,} rows")
print(f"    Columns: {list(df.columns)}")
print(f"    MO_TEXT sample: {df['MO_TEXT'].iloc[0]}")

OK  Loaded 500,000 rows
    Columns: ['IncidentID', 'CrimeType', 'CrimeSubType', 'Neighborhood', 'PropertyType', 'Latitude', 'Longitude', 'DateTime', 'SuspectCount', 'SuspectBuild', 'SuspectHeight', 'SuspectClothing', 'SuspectVehicle', 'VictimAge', 'VictimGender', 'VictimVulnerability', 'EntryMethod', 'Weapon', 'PropertyStolen', 'MO_TEXT']
    MO_TEXT sample: CCTV shows a single suspect, stocky build, average, wearing dark hoodie at Retail - Car park, Ashbrook at 1955hrs. Offender entered by bypass ignition wiring. Stereo, vehicle - bmw 3 series, laptop bag reported stolen. No witnesses at time of offence.


## 5. Load Embedding Model

`all-MiniLM-L6-v2` produces 384-dimensional embeddings. It is fast, well-supported and a standard baseline for semantic search benchmarks.

In [6]:
model = SentenceTransformer(MODEL_NAME)

print(f"OK  Model loaded: {MODEL_NAME}")
print(f"    Embedding dimensions: {model.get_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

OK  Model loaded: all-MiniLM-L6-v2
    Embedding dimensions: 384


## 6. Generate Embeddings

In [7]:
texts = df["MO_TEXT"].tolist()

start = time.time()

embeddings = model.encode(
    texts,
    batch_size = BATCH_SIZE,
    show_progress_bar = True,
    convert_to_numpy = True,
    normalize_embeddings = True,
)

elapsed = time.time() - start

print(f"\nOK  Embeddings generated")
print(f"    Shape:     {embeddings.shape}")
print(f"    Dtype:     {embeddings.dtype}")
print(f"    Time:      {elapsed:.1f}s ({len(texts) / elapsed:.0f} records/sec)")

Batches:   0%|          | 0/1954 [00:00<?, ?it/s]


OK  Embeddings generated
    Shape:     (500000, 384)
    Dtype:     float32
    Time:      511.9s (977 records/sec)


## 7. Sanity Check

Verify that normalized embeddings have unit norm and that similar incidents produce higher cosine similarity than dissimilar ones.

In [8]:
# Norms should all be 1.0 after normalization
norms = np.linalg.norm(embeddings, axis = 1)
print(f"Norm check - min: {norms.min():.4f}  max: {norms.max():.4f}  mean: {norms.mean():.4f}")

# Cosine similarity between two burglary incidents vs a burglary and an assault
burglary_idx = df[df["CrimeType"] == "Burglary"].index[:2].tolist()
assault_idx  = df[df["CrimeType"] == "Assault"].index[0]

sim_same  = float(np.dot(embeddings[burglary_idx[0]], embeddings[burglary_idx[1]]))
sim_cross = float(np.dot(embeddings[burglary_idx[0]], embeddings[assault_idx]))

print(f"\nCosine similarity (burglary vs burglary): {sim_same:.4f}")
print(f"Cosine similarity (burglary vs assault):  {sim_cross:.4f}")
print(f"\nBurglary A: {df['MO_TEXT'].iloc[burglary_idx[0]]}")
print(f"Burglary B: {df['MO_TEXT'].iloc[burglary_idx[1]]}")
print(f"Assault:    {df['MO_TEXT'].iloc[assault_idx]}")

Norm check - min: 1.0000  max: 1.0000  mean: 1.0000

Cosine similarity (burglary vs burglary): 0.4067
Cosine similarity (burglary vs assault):  0.4948

Burglary A: At approx 1632hrs, Multiple offenders described as medium build, tall, wearing face covering gained entry to Residence - Single Family in Hartley Cross. Property taken: handbag, tablet. No FO identified.
Burglary B: Between 1100hrs and 1300hrs, offender(s) suspected to have entered Commercial - Warehouse in Elmstead. Access via insecure garage door. Offender(s) made off with cash, jewellery, credit cards. Suspect vehicle: white Ford Focus. CCTV req.
Assault:    Suspect described as medium build, tall, wearing face covering involved in altercation at Public transport in Ravenscar at 0209hrs. Victim described as unknown gender, mid-40s. V flagged as elderly.


## 8. Save to Disk

In [9]:
ids = df["IncidentID"].to_numpy()

np.save(EMBEDDINGS_PATH, embeddings)
np.savetxt(IDS_PATH.replace(".npy", ".txt"), ids, fmt = "%s", encoding = "utf-8")

emb_size_mb = os.path.getsize(EMBEDDINGS_PATH) / 1024 / 1024

print(f"OK  Saved embeddings: {EMBEDDINGS_PATH} ({emb_size_mb:.1f} MB)")
print(f"OK  Saved IDs:        {IDS_PATH.replace('.npy', '.txt')}")

OK  Saved embeddings: data/nordvik_embeddings.npy (732.4 MB)
OK  Saved IDs:        data/nordvik_ids.txt


## 9. Verify Round-Trip

Load the files back from disk and confirm the shapes and values match.

In [10]:
embeddings_loaded = np.load(EMBEDDINGS_PATH)
ids_loaded        = np.loadtxt(IDS_PATH.replace(".npy", ".txt"), dtype = str, encoding = "utf-8")

assert embeddings_loaded.shape == embeddings.shape, "ERROR  Embedding shape mismatch"
assert np.allclose(embeddings_loaded, embeddings), "ERROR  Embedding values mismatch"
assert list(ids_loaded) == list(ids), "ERROR  ID mismatch"

print(f"OK  Round-trip verified")
print(f"    Embeddings: {embeddings_loaded.shape} {embeddings_loaded.dtype}")
print(f"    IDs:        {ids_loaded.shape} - first: {ids_loaded[0]}  last: {ids_loaded[-1]}")

OK  Round-trip verified
    Embeddings: (500000, 384) float32
    IDs:        (500000,) - first: NVK-000001  last: NVK-500000


---

## Embeddings Ready

Every chapter notebook loads these two files at the start:

```python
import numpy as np
import pandas as pd

df         = pd.read_parquet("data/nordvik_crimes.parquet")
embeddings = np.load("data/nordvik_embeddings.npy")
ids        = np.loadtxt("data/nordvik_ids.txt", dtype = str, encoding = "utf-8")
```

| File | Shape | Description |
|---|---|---|
| `nordvik_embeddings.npy` | `(n_records, 384)` | Normalized float32 vectors |
| `nordvik_ids.txt` | `(n_records,)` | IncidentID strings in matching row order |